# Part 3 — Agent Skills: teaching an agent how, not just what

In Parts 1 and 2, we gave Claude a fixed set of MOFA-specific tools. Claude could decide which of those tools to use, but it could only do the things we had explicitly made available.

In this part, we change that setup. Instead of giving the agent one specialised tool for each operation, we use a **coding agent** with a few general-purpose capabilities: it can read files, write files, and run commands. With those alone, it can inspect the repository, find `mofa_tools.py`, and run the analysis itself.

That gives the agent much more freedom. But it also creates a new problem: **how does it know how we want this particular analysis to be carried out?**

That is where **Agent Skills** come in. A skill gives the agent reusable instructions for a particular kind of task: it can define things such as which resouces to use and where to find them, which steps to follow, or how should be the format of the final result.

In this notebook, we will give the coding agent the same MOFA analysis task twice: once with a MOFA-specific skill, and once without it. The agent has the same general capabilities in both cases. What changes is the guidance available to it.

## Learning objectives

By the end of this notebook you should be able to:
- Explain the difference between giving an agent a fixed set of specialised tools and giving it general-purpose coding capabilities.
- Explain what an **Agent Skill** is and why it can be useful when an agent has many possible ways to approach a task.
- Compare the same task with and without a skill, and identify what the skill changes.
- Recognise the difference between **capability** — what the agent is able to do — and **procedural guidance** — how we want it to do it.

## 0. From specialised tools to a coding agent
In Parts 1 and 2, we carefully chose the functions Claude was allowed to call. For example, if we wanted Claude to inspect factor weights, we exposed a function specifically for that purpose.

A coding agent works differently. Instead of receiving a list of task-specific functions, it receives a small set of much more general tools. For example, it may be able to:
- read a file,
- write or edit a file,
- run a command in the terminal

If the agent can read the repository, and run `bash` commands, it can find `src/mofa_tools.py`, inspect the available functions, import them, and execute them itself. We no longer need o turn every possible operation into a separate tool first. This is the kind of setup used by coding agents such as Claude Code, Codex, Cursor, OpenHands and Pi.

The advantage is flexibility: the agent can work with code and files that were not individually prepared for it in advance.

The disadvantage is that there are now many more possible ways to perform the same task. The agent may be able to run the analysis, but that does not mean it automatically knows how we want the analysis to be done.

## 1. The new problem: how do we tell the agent how to do the task?
Consider our MOFA analysis.

A general-purpose coding agent can inspect the repository and run Python. But from those capabilities alone, it does not know things such as:

- that it should load the cached MOFA model rather than fit a new one,
- which functions are the intended interface to the analysis,
- what evidence should support its conclusions,
- how cautious it should be when interpreting biological results,
- how we want the final answer to be structured.

We could put all of those instructions into the prompt every time we ask a question. But if we repeatedly perform the same kind of task, it is more useful to save those instructions once and reuse them.

That reusable set of instructions is a **Skill**.

### Agent Skills & `SKILL.md`
An Agent Skill is a reusable set of instructions describing how an agent should approach a particular kind of task. 

A skill is stored as a folder. At its centre is a file called SKILL.md, which contains a short description of when the skill is relevant and the instructions for carrying out the task. The file starts with a small amount of metadata — its name and description — followed by the actual instructions. A skill can also include supporting `scripts/`, `references/` or other `assets/` when needed.

For our MOFA analysis in this notebook, those instructions include using the cached model rather than refitting it, working through the existing functions in `src/mofa_tools.py`, grounding conclusions in results from the analysis, following a consistent answer structure, and being cautious when making biological interpretations.

These instructions do not give the agent any new capabilities. The agent can already read files and run Python. The skill provides **procedural knowledge**: guidance about how those capabilities should be used for this particular task.

One useful feature of Agent Skills is that the agent does not need to load every available skill in full. Instead, skills are revealed progressively:

1. **Discovery:** the agent initially sees only each skill's `name` and `description`.
2. **Activation:** if a skill looks relevant to the current task, the agent loads its full `SKILL.md`.
3. **Execution:** it follows those instructions and can use any supporting scripts or references included with the skill if needed.

This means an agent can have many skills available without having to place all of their instructions into its context at once.

Because the format is standardised, the same skill can also be reused across different compatible coding agents. Agent Skills were introduced by Anthropic as an [open standard](https://agentskills.io/specification) and are supported by several agent systems, including Claude Code, Cursor, Codex and Pi.

### Where does the harness fit?
The language model still does not directly read files or execute commands. As in the previous parts, those actions are carried out by the **harness** around the model.

With a coding agent, the harness provides general-purpose tools such as file access and `bash` commands, runs the agent loop, and makes skills available to the model.

In this notebook, we use Pi as that harness.

## 2. How skills overlap with tools and MCP

We have now seen three different ways of extending how an LLM agent can work.

| | What it adds | Form | Answers… |
|---|---|---|---|
| Tools (Part1) | a new capability | a typed function the model calls | what can the agent do? |
| MCP (Part2) | the same capabilities, portably | a standard server for tools/resources/prompts | where do they live, who reuses them? |
| Skills (Part3) | know-how / workflow | a `SKILL.md` folder of instructions | how should it do it, and when? |

<br>

> Tools & MCP determine what an agent can do. Skills help determine **how it should do the task**

These ideas can also be combined: a skill can tell an agent to use particular tools, including tools provided through MCP. The difference is whether we are giving the agent a *capability* (a tool) or giving it *guidance for using the capabilities it already has* (a skill).

## 3. The coding agent we will use: Pi

For this notebook, we use **Pi**, a command-line coding agent ([pi.dev](https://pi.dev)). Pi gives Claude general-purpose tools for interacting with the project, including reading files and running commands. 

Pi also supports Agent Skills, allowing us to provide the MOFA-specific instructions in `SKILL.md`. This means Claude can inspect the repository and run the MOFA analysis without us exposing each MOFA function as an individual tool. We use Pi here as one example, but the same skill could also be used by other compatible coding agents (Claude Code, Codex, etc.).

Pi is already installed in the workshop environment. Elsewhere, it can be installed with:

```bash
npm install -g --ignore-scripts @earendil-works/pi-coding-agent
# or: curl -fsSL https://pi.dev/install.sh | sh
```

## 4. Setup and the skill we'll hand to Pi

This cell performs the same basic setup as Parts 1 and 2, and additionally locates Pi and our `SKILL.md` file. Because Pi will run Python commands itself, we also make it use the **same workshop environment as this notebook**, where the required packages and `src.mofa_tools` are already available.

In [1]:

import os, sys, shutil, subprocess
from pathlib import Path
from dotenv import load_dotenv

# Same setup as Parts 1 & 2
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not found (.env in project root)."
MODEL = "anthropic/claude-haiku-4-5"

# Part 3 also needs to know where the skill file is
SKILL_PATH = PROJECT_ROOT / "skills" / "mofa-multiomics-agent.SKILL.md"

# Find the Pi command-line program installed in the workshop environment
PI = shutil.which("pi") or str(Path.home() / ".local" / "bin" / "pi")
assert Path(PI).exists(), f"Pi CLI not found at {PI}. Install it (see https://pi.dev)."

# Pi will later run Python through its bash tool. 
# We make those commands use the same Python environment as this notebook, 
# so Pi has access to the same packages and project code, including src.mofa_tools.
ENV_BIN = str(Path(sys.executable).parent)
PI_ENV = {**os.environ, "PATH": f"{ENV_BIN}:{Path(PI).parent}:{os.environ.get('PATH', '')}"}
print("python    :", sys.executable)

python    : /home/eidf128/eidf128/nfabrega_big_eidf/.conda/envs/eccb2026/bin/python


### Looking at the skill
Before using the skill, let us look at the instructions we are actually giving the agent. 

The next cell simply prints `SKILL.md`. Notice that it does not add new analytical capabilities: it provides instructions for how Pi should use the code already available, what evidence it should rely on, and how it should report the results.

In [2]:
print(SKILL_PATH.read_text())

---
name: mofa-multiomics-agent
description: Analyse a fitted MOFA multi-omics model of TCGA breast-cancer data — interpret latent factors, associate factors with PAM50 subtype, rank factor drivers, predict subtype from factors, and produce diagnostic plots. Use when a task involves MOFA factors, variance explained (R2), factor<->subtype association, factor weights/drivers, or multi-omics subtype prediction. Enforces tool-grounded evidence, loading the cached model (never re-fitting), a fixed answer format, and biomedical caution.
---

# MOFA Multi-Omics Agent Skill

## Purpose

Use this skill when answering questions about a fitted MOFA model of TCGA
breast-cancer multi-omics data (transcriptomics, proteomics, methylation) with
PAM50 subtype labels: factor interpretation, factor↔subtype association, factor
drivers (weights), subtype prediction from factors, and diagnostic plots.

## Behaviour

- Always compute results with the repo's functions before making quantitative
  claims about

## 5. Run Pi as a harness, with the skill
We are now ready to give Pi a task. If we were using Pi directly from a terminal, a command would look like this:

```bash
pi -p "<task>" --skill skills/mofa-multiomics-agent.SKILL.md \
   --model anthropic/claude-haiku-4-5 --approve
```

Here:

- `-p "<task>"` gives Pi the task to perform;
- `--skill ...` loads our MOFA skill;
- `--model ...` specifies the language model;
- `--approve` allows Pi to work with the project files without stopping for confirmation.

Once started, Pi runs the agent loop itself. Claude can choose when to read files, run `bash` commands, edit or write files, inspect the results, and continue until the task is complete.

### Launching Pi prom Python
Because we are working inside a notebook rather than typing commands into a terminal, the next cell wraps this command in a Python function called `run_pi()`.

The function uses Python's `subprocess` module, which simply allows Python to start another program and collect its output. Here, that other program is Pi.

The `with_skill` argument determines whether the Pi command includes our `--skill` file or runs with `--no-skills`.

In [3]:
def run_pi(prompt: str, with_skill: bool, model: str = MODEL, timeout: int = 600):
    """Invoke the Pi coding agent headlessly and return its final text answer."""
    cmd = [PI, "-p", prompt, "--model", model, "--approve"]
    cmd += ["--skill", str(SKILL_PATH)] if with_skill else ["--no-skills"]
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT, env=PI_ENV,
                          capture_output=True, text=True, timeout=timeout)
    if proc.returncode != 0:
        print("[pi stderr]\n", proc.stderr[-2000:])
    return proc.stdout.strip()

### Trying some tasks

We can now give Pi broader tasks and let it decide how to use the repository and its general-purpose tools to complete them.

In [ ]:
PIPELINE_TASK = (
    "Run the full breast-cancer subtype-prediction pipeline on the fitted MOFA "
    "model (load the cached model, do not re-fit): report which factors are "
    "active, which factor is most associated with PAM50 subtype, and the held-out "
    "classification performance. Then give a one-paragraph interpretation.")
print(run_pi(PIPELINE_TASK, with_skill=True))

In [ ]:
#################################################
# Write a task asking Pi to generate the standard MOFA diagnostic plots into
# outputs/, and say which single plot best supports the claim that MOFA captures
# breast-cancer subtype biology, and why.
#
### WRITE YOUR CODE HERE ###
#
#################################################

## 6. Running the same task without the skill

Now we run the same `PIPELINE_TASK` again, using the same model and the same general-purpose tools, but without loading the skill.

Pi can still inspect the repository, run Python and use src/mofa_tools.py. Its **capabilities therefore remain the same**; what is removed is the additional procedural guidance in `SKILL.md`.


Same agent, model, task, but `--no-skills`. Pi can still find
`src/mofa_tools.py` by exploring the repo, so the capability is unchanged.
What changes is behaviour: without the skill it has no instruction to load
the cached model (it might try to re-fit), to ground every claim in tool output,
to use the fixed answer format, or to add biomedical caveats.

In [ ]:
print(run_pi(PIPELINE_TASK, with_skill=False))

compare the two outputs. What does the skill change about the way the model approaches and reports the task?

## Reflection

- Compare the run `with_skill`  to the run without it. What differences do you notice in how the task is performed?
- Did the skill change what Pi could do, or how it used its existing capabilities?
- Here, we explicitly tell Pi which skill to use as there's only a handful. How would this work if Pi had 50 skills available?
- if the MOFA functions were accessed through MCP instead of directly through MCP instead of directly through Python, which parts of the skill would stay the same, and which would need to change?